# 10 — Monitoramento & Deploy

Este notebook implementa o pipeline de monitoramento contínuo do modelo champion e prepara o deploy via OCI Model Catalog:

- **PSI Monitoring** — estabilidade da distribuição de scores ao longo do tempo
- **Feature Drift** — detecção de drift nas top 20 features
- **Backtesting** — comparação ensemble vs baseline por SAFRA
- **Monitoramento dos Modelos Base** — PSI e concordância entre modelos do ensemble
- **Registro no OCI Model Catalog** — catalogação com metadados e métricas
- **Relatório Final** — consolidação do status operacional

In [ ]:
import numpy as np
import pandas as pd
import pickle
import json
import os
from datetime import datetime
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from config import *
from utils import *

## PSI Monitoring — Estabilidade do Score

O **Population Stability Index (PSI)** mede a estabilidade da distribuição de scores entre o período de treino e cada safra OOT. Interpretação:
- **PSI < 0.10** — Estável (OK)
- **0.10 <= PSI < 0.25** — Mudança moderada (WARNING)
- **PSI >= 0.25** — Mudança significativa (RETRAIN)

In [ ]:
# ---------------------------------------------------------------------------
# Carregar scores e modelo
# ---------------------------------------------------------------------------
scores_path = os.path.join(ARTIFACT_DIR, 'scores', 'scores_full.parquet')
scores_df = pd.read_parquet(scores_path)
print(f"Scores carregados: {len(scores_df):,} registros")

# Separar scores por período
train_scores = scores_df[scores_df['SAFRA'].isin(TRAIN_SAFRAS)]['SCORE'].values
print(f"Scores treino: {len(train_scores):,}")

# PSI por SAFRA OOT
print("\n=== PSI por SAFRA (Score) ===")
print(f"{'SAFRA':<12} {'PSI':>8} {'Status':>10}")
print('-' * 32)

psi_results = []
for safra in sorted(scores_df['SAFRA'].unique()):
    safra_scores = scores_df[scores_df['SAFRA'] == safra]['SCORE'].values
    psi_val = compute_psi(train_scores, safra_scores)
    status = psi_alert(psi_val)
    psi_results.append({'safra': safra, 'psi': psi_val, 'status': status})
    print(f"{safra:<12} {psi_val:>8.4f} {status:>10}")

## Feature Drift — Top 20 Features

Para cada uma das **20 features mais importantes**, calculamos o PSI entre a distribuição no treino e no OOT. Features com PSI > 0.25 são sinalizadas como instáveis.

In [ ]:
# ---------------------------------------------------------------------------
# Feature Drift — Top 20 Features
# ---------------------------------------------------------------------------
fs_path = LOCAL_DATA_PATH
top_features = SELECTED_FEATURES[:20]
cols_to_load = ['SAFRA'] + top_features
df_fs = pd.read_parquet(fs_path, columns=cols_to_load)

df_train = df_fs[df_fs['SAFRA'].isin(TRAIN_SAFRAS)]
df_oot = df_fs[df_fs['SAFRA'].isin(OOT_SAFRAS)]

print("=== Feature Drift (Top 20 Features) ===")
print(f"{'Feature':<40} {'PSI':>8} {'Status':>10}")
print('-' * 60)

drift_results = []
drift_flags = []
for feat in top_features:
    train_vals = df_train[feat].dropna().values.astype(float)
    oot_vals = df_oot[feat].dropna().values.astype(float)

    if len(train_vals) == 0 or len(oot_vals) == 0:
        psi_val = np.nan
        status = 'N/A'
    else:
        psi_val = compute_psi(train_vals, oot_vals)
        status = psi_alert(psi_val)

    drift_results.append({'feature': feat, 'psi': psi_val, 'status': status})
    if psi_val is not None and not np.isnan(psi_val) and psi_val > 0.25:
        drift_flags.append(feat)
    print(f"{feat:<40} {psi_val:>8.4f} {status:>10}")

if drift_flags:
    print(f"\nALERTA: {len(drift_flags)} feature(s) com drift significativo (PSI > 0.25):")
    for f in drift_flags:
        print(f"  - {f}")
else:
    print("\nNenhuma feature com drift significativo.")

## Backtesting: Ensemble vs Baseline por SAFRA

Avaliamos o desempenho do modelo ensemble em cada SAFRA e comparamos com um baseline (modelo individual ou heurística).

In [ ]:
# ---------------------------------------------------------------------------
# Backtesting por SAFRA
# ---------------------------------------------------------------------------
from sklearn.metrics import roc_auc_score

# Carregar ensemble
champion_path = os.path.join(ARTIFACT_DIR, 'models', 'champion_ensemble.pkl')
with open(champion_path, 'rb') as fh:
    ensemble_model = pickle.load(fh)

# Carregar baseline (primeiro modelo individual)
model_dir = os.path.join(ARTIFACT_DIR, 'models')
baseline_files = sorted([f for f in os.listdir(model_dir) if f.endswith('.pkl') and 'ensemble' not in f])
with open(os.path.join(model_dir, baseline_files[0]), 'rb') as fh:
    baseline_model = pickle.load(fh)
baseline_name = baseline_files[0].replace('.pkl', '')

# Feature Store com target
df_full = pd.read_parquet(
    LOCAL_DATA_PATH,
    columns=['SAFRA'] + SELECTED_FEATURES + [TARGET]
)

print(f"=== Backtesting: Ensemble vs {baseline_name} ===")
print(f"{'SAFRA':<10} {'KS_Ens':>8} {'AUC_Ens':>8} {'KS_Base':>8} {'AUC_Base':>9} {'Delta_KS':>9}")
print('-' * 55)

backtest_results = []
for safra in sorted(df_full['SAFRA'].unique()):
    df_safra = df_full[df_full['SAFRA'] == safra].dropna(subset=[TARGET])
    if len(df_safra) < 100:
        continue

    X_safra = df_safra[SELECTED_FEATURES]
    y_safra = df_safra[TARGET]

    # Ensemble
    pred_ens = ensemble_model.predict_proba(X_safra)[:, 1]
    ks_ens = compute_ks(y_safra, pred_ens)
    auc_ens = roc_auc_score(y_safra, pred_ens)

    # Baseline
    pred_base = baseline_model.predict_proba(X_safra)[:, 1]
    ks_base = compute_ks(y_safra, pred_base)
    auc_base = roc_auc_score(y_safra, pred_base)

    delta_ks = ks_ens - ks_base
    backtest_results.append({
        'safra': safra, 'ks_ensemble': ks_ens, 'auc_ensemble': auc_ens,
        'ks_baseline': ks_base, 'auc_baseline': auc_base, 'delta_ks': delta_ks
    })
    print(f"{safra:<10} {ks_ens:>8.4f} {auc_ens:>8.4f} {ks_base:>8.4f} {auc_base:>9.4f} {delta_ks:>+9.4f}")

## Monitoramento dos Modelos Base do Ensemble

Para cada modelo base do ensemble, calculamos o PSI das predições (treino vs OOT) e verificamos a taxa de concordância entre os modelos.

In [ ]:
# ---------------------------------------------------------------------------
# Monitoramento dos modelos base
# ---------------------------------------------------------------------------
df_train_data = df_full[df_full['SAFRA'].isin(TRAIN_SAFRAS)]
df_oot_data = df_full[df_full['SAFRA'].isin(OOT_SAFRAS)]

X_train = df_train_data[SELECTED_FEATURES]
X_oot = df_oot_data[SELECTED_FEATURES]

if hasattr(ensemble_model, 'base_models'):
    base_models = ensemble_model.base_models
else:
    # Fallback: carregar modelos individuais
    base_models = {}
    for mf in baseline_files:
        name = mf.replace('.pkl', '')
        with open(os.path.join(model_dir, mf), 'rb') as fh:
            base_models[name] = pickle.load(fh)

print("=== PSI dos Modelos Base (Predições Treino vs OOT) ===")
print(f"{'Modelo':<30} {'PSI':>8} {'Status':>10}")
print('-' * 50)

base_preds_oot = {}
for name, model in base_models.items():
    pred_train = model.predict_proba(X_train)[:, 1]
    pred_oot = model.predict_proba(X_oot)[:, 1]
    psi_val = compute_psi(pred_train, pred_oot)
    status = psi_alert(psi_val)
    base_preds_oot[name] = (pred_oot > 0.5).astype(int)
    print(f"{name:<30} {psi_val:>8.4f} {status:>10}")

# Taxa de concordância
if len(base_preds_oot) > 1:
    model_list = list(base_preds_oot.keys())
    print(f"\n=== Taxa de Concordância entre Modelos Base (OOT) ===")
    for i in range(len(model_list)):
        for j in range(i + 1, len(model_list)):
            agreement = np.mean(base_preds_oot[model_list[i]] == base_preds_oot[model_list[j]])
            print(f"  {model_list[i]} vs {model_list[j]}: {agreement:.2%}")

## Registro no OCI Model Catalog

Tentamos registrar o modelo no **OCI Data Science Model Catalog** com metadados customizados (métricas, features, safras). Em caso de limitações do ambiente trial, documentamos a tentativa.

In [ ]:
# ---------------------------------------------------------------------------
# Registro no OCI Model Catalog
# ---------------------------------------------------------------------------
def document_trial_limitation(reason: str, metadata: dict):
    """Documenta limitação do ambiente trial e salva metadados localmente."""
    doc = {
        'status': 'TRIAL_LIMITATION',
        'reason': reason,
        'metadata': metadata,
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'note': 'Modelo não registrado no OCI Model Catalog devido a limitações do ambiente trial. '
                'Metadados salvos localmente para registro manual posterior.'
    }
    doc_path = os.path.join(ARTIFACT_DIR, 'model_catalog_registration.json')
    with open(doc_path, 'w') as fh:
        json.dump(doc, fh, indent=2)
    print(f"Documentação salva em: {doc_path}")
    return doc

# Metadata para o catálogo
champion_meta_path = os.path.join(ARTIFACT_DIR, 'champion_metadata.json')
if os.path.exists(champion_meta_path):
    with open(champion_meta_path, 'r') as fh:
        champion_metadata = json.load(fh)
else:
    champion_metadata = {}

catalog_metadata = {
    'model_name': 'fpd-credit-risk-ensemble',
    'model_version': champion_metadata.get('strategy', 'v1.0'),
    'framework': 'scikit-learn',
    'target': TARGET,
    'n_features': len(SELECTED_FEATURES),
    'metrics': champion_metadata.get('metrics', {}),
    'train_safras': TRAIN_SAFRAS,
    'oot_safras': OOT_SAFRAS,
    'feature_list': SELECTED_FEATURES[:20],
    'created_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
}

try:
    import oci
    from oci.data_science.models import CreateModelDetails

    config = oci.config.from_file()
    ds_client = oci.data_science.DataScienceClient(config)

    custom_metadata = [
        {'key': k, 'value': str(v)[:255]}
        for k, v in catalog_metadata.items()
        if isinstance(v, (str, int, float))
    ]

    model_details = CreateModelDetails(
        compartment_id=config.get('compartment_id', NAMESPACE),
        project_id=os.environ.get('OCI_PROJECT_ID', 'hackathon-pod-academy'),
        display_name='FPD Credit Risk Ensemble — Hackathon PoD Academy',
        description=f"Ensemble model ({champion_metadata.get('strategy', 'N/A')}) para predição de First Payment Default.",
        custom_metadata_list=custom_metadata,
    )

    response = ds_client.create_model(model_details)
    print(f"Modelo registrado no OCI Model Catalog!")
    print(f"  Model OCID: {response.data.id}")
    print(f"  Status: {response.data.lifecycle_state}")

except ImportError:
    print("OCI SDK não disponível. Documentando limitação do ambiente trial.")
    document_trial_limitation(
        reason='OCI SDK (oci) não instalado no ambiente atual.',
        metadata=catalog_metadata
    )

except Exception as e:
    print(f"Erro ao registrar no OCI Model Catalog: {e}")
    document_trial_limitation(
        reason=str(e),
        metadata=catalog_metadata
    )

## Relatorio Final

Consolidação de todos os indicadores de monitoramento em um relatório JSON. O status geral segue a regra:
- **STABLE** — todos os indicadores dentro dos limites
- **WARNING** — algum indicador com PSI entre 0.10 e 0.25
- **RETRAIN** — algum indicador com PSI >= 0.25 ou degradação significativa de KS

In [ ]:
# ---------------------------------------------------------------------------
# Relatório Final de Monitoramento
# ---------------------------------------------------------------------------
# Determinar status geral
psi_statuses = [r['status'] for r in psi_results]
drift_statuses = [r['status'] for r in drift_results if r['status'] != 'N/A']
all_statuses = psi_statuses + drift_statuses

if 'RETRAIN' in all_statuses:
    overall_status = 'RETRAIN'
elif 'WARNING' in all_statuses:
    overall_status = 'WARNING'
else:
    overall_status = 'STABLE'

monitoring_report = {
    'report_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'overall_status': overall_status,
    'model': {
        'name': 'fpd-credit-risk-ensemble',
        'strategy': champion_metadata.get('strategy', 'N/A'),
        'metrics': champion_metadata.get('metrics', {}),
    },
    'score_psi': [
        {'safra': r['safra'], 'psi': round(r['psi'], 4), 'status': r['status']}
        for r in psi_results
    ],
    'feature_drift': [
        {'feature': r['feature'], 'psi': round(r['psi'], 4) if not np.isnan(r['psi']) else None, 'status': r['status']}
        for r in drift_results
    ],
    'features_with_drift': drift_flags,
    'backtesting': backtest_results,
    'recommendations': [],
}

# Recomendações automáticas
if overall_status == 'RETRAIN':
    monitoring_report['recommendations'].append(
        'RETRAIN: Drift significativo detectado. Recomendado retreinar o modelo com dados recentes.'
    )
if drift_flags:
    monitoring_report['recommendations'].append(
        f'FEATURE_DRIFT: {len(drift_flags)} feature(s) com drift > 0.25. Investigar causa raiz.'
    )
if backtest_results:
    avg_delta = np.mean([r['delta_ks'] for r in backtest_results])
    if avg_delta < -0.05:
        monitoring_report['recommendations'].append(
            f'DEGRADACAO: KS médio do ensemble caiu {abs(avg_delta):.4f} vs baseline. Avaliar retreino.'
        )

if not monitoring_report['recommendations']:
    monitoring_report['recommendations'].append('Modelo estável. Nenhuma ação necessária.')

# Salvar relatório
monitoring_dir = os.path.join(ARTIFACT_DIR, 'monitoring')
os.makedirs(monitoring_dir, exist_ok=True)
report_path = os.path.join(monitoring_dir, 'monitoring_report.json')
with open(report_path, 'w') as fh:
    json.dump(monitoring_report, fh, indent=2, default=str)

# Sumário
print("=" * 60)
print("         RELATÓRIO FINAL DE MONITORAMENTO")
print("=" * 60)
print(f"  Data:            {monitoring_report['report_date']}")
print(f"  Status Geral:    {overall_status}")
print(f"  Modelo:          {monitoring_report['model']['name']}")
print(f"  Estratégia:      {monitoring_report['model']['strategy']}")
print(f"  Métricas:        {json.dumps(monitoring_report['model']['metrics'])}")
print(f"  Features c/ drift: {len(drift_flags)}")
print(f"  SAFRAs avaliadas:  {len(psi_results)}")
print(f"\n  Recomendações:")
for rec in monitoring_report['recommendations']:
    print(f"    - {rec}")
print(f"\n  Relatório salvo em: {report_path}")
print("=" * 60)

log(f"09_monitoring | Status: {overall_status} | Drift features: {len(drift_flags)} | Report: {report_path}")